# DATATHON 2026 - Phần 1: Câu hỏi Trắc nghiệm

**Mục tiêu:** Trả lời 10 câu hỏi trắc nghiệm dựa trên phân tích dữ liệu

**Điểm số:** 20 điểm (2 điểm mỗi câu đúng)

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Setup paths
DATA_DIR = '../dataset/'

# Load all required CSV files
orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
products = pd.read_csv(DATA_DIR + 'products.csv')
returns = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
geography = pd.read_csv(DATA_DIR + 'geography.csv')
payments = pd.read_csv(DATA_DIR + 'payments.csv')
sales = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])

print(f"Orders shape: {orders.shape}")
print(f"Products shape: {products.shape}")
print(f"Returns shape: {returns.shape}")

✅ All data loaded successfully!
Orders shape: (646945, 8)
Products shape: (2412, 8)
Returns shape: (39939, 7)


## Q1: Inter-order Gap

**Câu hỏi:** Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu?

In [28]:
orders_sorted = orders.sort_values(['customer_id', 'order_date']).reset_index(drop=True)

# Count orders per customer
order_counts = orders_sorted.groupby('customer_id').size()
multi_order_customers = order_counts[order_counts > 1].index

# Filter customers with multiple orders
multi_orders = orders_sorted[orders_sorted['customer_id'].isin(multi_order_customers)]

# Calculate gaps
gaps = []
for customer_id in multi_order_customers:
    customer_orders = multi_orders[multi_orders['customer_id'] == customer_id]['order_date'].sort_values()
    customer_gaps = customer_orders.diff().dt.days.dropna()
    gaps.extend(customer_gaps.values)

gaps = np.array(gaps)
gaps = gaps[gaps > 0]  

median_gap = np.median(gaps)
print(f"Q1: Median inter-order gap = {median_gap:.0f} days")
print(f"Mean: {np.mean(gaps):.2f}")
print(f"Median: {median_gap:.0f}")
print(f"Min: {np.min(gaps):.0f}, Max: {np.max(gaps):.0f}")

Q1: Median inter-order gap = 146 days
Mean: 288.22
Median: 146
Min: 1, Max: 3785


**Đáp án Q1:** C) 180 ngày (vì 146 ngày gần 180 ngày hơn 90 ngày) 

## Q2: Product Segment Profitability

**Câu hỏi:** Phân khúc sản phẩm (segment) nào có tỷ suất lợi nhuận gộp trung bình cao nhất?

In [29]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
segment_margin = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

print("Q2: Gross Profit Margin by Segment:")
print(segment_margin)
print(f"\nAnswer: {segment_margin.idxmax()} has the highest margin: {segment_margin.max():.4f}")

Q2: Gross Profit Margin by Segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Answer: Standard has the highest margin: 0.3134


**Đáp án Q2:** D) Standard

## Q3: Return Reasons for Streetwear

**Câu hỏi:** Lý do trả hàng nào xuất hiện nhiều nhất cho sản phẩm Streetwear?

In [30]:
# Join returns with products
returns_products = returns.merge(products[['product_id', 'category']], on='product_id', how='left')

# Filter for Streetwear
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']

return_reasons = streetwear_returns['return_reason'].value_counts()
print("Q3: Return Reasons for Streetwear Category:")
print(return_reasons)
print(f"\nMost common reason: {return_reasons.index[0]}")

Q3: Return Reasons for Streetwear Category:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Most common reason: wrong_size


**Đáp án Q3:** B) wrong_size

## Q4: Traffic Source Bounce Rate

**Câu hỏi:** Nguồn truy cập nào có tỷ lệ thoát trung bình thấp nhất?

In [38]:
traffic_bounce = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print("Q4: Average Bounce Rate by Traffic Source:")
print(traffic_bounce)
print(f"\nLowest bounce rate: {traffic_bounce.index[0]}")

Q4: Average Bounce Rate by Traffic Source:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Lowest bounce rate: email_campaign


**Đáp án Q4:** C) email_campaign

## Q5: Promotion Usage Percentage

**Câu hỏi:** Tỷ lệ phần trăm các dòng trong order_items có áp dụng khuyến mãi là bao nhiêu?

In [32]:
total_items = len(order_items)
promo_items = order_items['promo_id'].notna().sum()
promo_percentage = (promo_items / total_items) * 100

print(f"Q5: Promotion Application:")
print(f"promo percentage: {promo_percentage:.2f}%")

Q5: Promotion Application:
promo percentage: 38.66%


**Đáp án Q5:** C) 39%

## Q6: Age Group Order Frequency

**Câu hỏi:** Nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất?

In [39]:
# Join customers with orders
customers_orders = customers.merge(orders, on='customer_id')

# Filter for non-null age_group
customers_orders = customers_orders[customers_orders['age_group'].notna()]

# Orders per customer per age group
age_group_orders = customers_orders.groupby(['age_group', 'customer_id']).size().reset_index(name='order_count')
avg_orders_by_age = age_group_orders.groupby('age_group')['order_count'].mean().sort_values(ascending=False)

print("Q6: Average Orders per Customer by Age Group:")
print(avg_orders_by_age)
print(f"\nHighest average: {avg_orders_by_age.index[0]}")

Q6: Average Orders per Customer by Age Group:
age_group
55+      7.268731
45-54    7.220264
35-44    7.206159
25-34    7.112230
18-24    7.068577
Name: order_count, dtype: float64

Highest average: 55+


**Đáp án Q6:** A) 55+

## Q7: Region Revenue

**Câu hỏi:** Vùng nào tạo ra tổng doanh thu cao nhất?

In [40]:
# Join orders with geography using zip
orders_geo = orders.merge(geography, on='zip', how='left')

# Join with sales using date
orders_sales = orders_geo[['order_id', 'order_date', 'region']].copy()
sales_daily = sales[['Date']].copy()
sales_daily['daily_revenue'] = sales['Revenue']

# Alternative: sum revenue by region from orders
orders_payment = orders_geo.merge(payments, on='order_id', how='left')
region_revenue = orders_payment.groupby('region')['payment_value'].sum().sort_values(ascending=False)

print("Q7: Total Revenue by Region:")
print(region_revenue)
print(f"\nHighest revenue region: {region_revenue.index[0]}")

Q7: Total Revenue by Region:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64

Highest revenue region: East


**Đáp án Q7:** C) East

## Q8: Cancelled Orders Payment Method

**Câu hỏi:** Phương thức thanh toán nào được sử dụng nhiều nhất trong các đơn bị hủy?

In [41]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']['order_id'].values
cancelled_payments = payments[payments['order_id'].isin(cancelled_orders)]

payment_methods = cancelled_payments['payment_method'].value_counts()
print("Q8: Payment Methods for Cancelled Orders:")
print(payment_methods)
print(f"\nMost common: {payment_methods.index[0]}")

Q8: Payment Methods for Cancelled Orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Most common: credit_card


**Đáp án Q8:** A) credit_card

## Q9: Return Rate by Product Size

**Câu hỏi:** Kích thước sản phẩm nào có tỷ lệ trả hàng cao nhất?

In [42]:
# Join order_items with returns and products
items_returns = order_items.merge(products[['product_id', 'size']], on='product_id')
returns_count = returns.merge(products[['product_id', 'size']], on='product_id').groupby('size').size()
items_count = items_returns.groupby('size').size()

return_rate = (returns_count / items_count * 100).sort_values(ascending=False)
print("Q9: Return Rate by Size:")
print(return_rate)
print(f"\nHighest return rate: {return_rate.index[0]}")

Q9: Return Rate by Size:
size
S     5.651527
L     5.624978
M     5.566010
XL    5.520010
dtype: float64

Highest return rate: S


**Đáp án Q9:** A) S

## Q10: Payment Installment Average

**Câu hỏi:** Kế hoạch trả góp nào có giá trị thanh toán trung bình cao nhất?

In [43]:
installment_avg = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print("Q10: Average Payment Value by Installment Plan:")
print(installment_avg)
print(f"\nHighest average: {installment_avg.index[0]}")

Q10: Average Payment Value by Installment Plan:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

Highest average: 6


**Đáp án Q10:** C) 6 kỳ

## Summary of Answers

Tóm tắt các đáp án cho 10 câu hỏi trắc nghiệm:

| Câu | Đáp án |
|-----|--------|
| Q1  | C      |
| Q2  | D      |
| Q3  | B      |
| Q4  | C      |
| Q5  | C      |
| Q6  | A      |
| Q7  | C      |
| Q8  | A      |
| Q9  | A      |
| Q10 | C      |